# Recommandation de films par similarité de contenu

Pas de LLM ici : on construit, pour chaque film, un "soup" de texte (genres + tags + casting + tagline), on le vectorise avec `TfidfVectorizer`, puis on calcule une similarité cosinus entre tous les films. Pour un film donné, les 5 films les plus proches deviennent la recommandation.

Données lues directement depuis `data/cinedata.db` (pas via l'API — aucune route n'est prévue pour ça).

## 1. Configuration et chargement des données

In [1]:
import sqlite3
from pathlib import Path

import yaml
import pandas as pd

with open("../config.yaml", "r", encoding="utf-8") as f:
    CONFIG = yaml.safe_load(f)

CHEMIN_DB = "../" + CONFIG["database"]["chemin"]
DOSSIER_SQL = "../" + str(Path(CONFIG["database"]["schema"]).parent)
NOM_FICHIER_SQL = CONFIG["recommandation"]["sql"]
TOP_N = CONFIG["recommandation"]["top_n"]

print(f"Base : {CHEMIN_DB}")
print(f"Requête : {DOSSIER_SQL}/{NOM_FICHIER_SQL}")
print(f"Nombre de recommandations : {TOP_N}")

Base : ../data/cinedata.db
Requête : ../sql/recommandation_donnees.sql
Nombre de recommandations : 5


In [11]:
connexion = sqlite3.connect(CHEMIN_DB)
connexion.row_factory = sqlite3.Row # Permet de récupérer les résultats sous forme de dictionnaire

with open(f"{DOSSIER_SQL}/{NOM_FICHIER_SQL}", "r", encoding="utf-8") as f:
    requete = f.read()

lignes = connexion.execute(requete).fetchall()
print(f"1er résultat : {dict(lignes[0])}")

df = pd.DataFrame([dict(ligne) for ligne in lignes])
connexion.close()

print(f"{len(df)} film(s) chargé(s).")
df.head()

1er résultat : {'id': 11, 'title': 'Star Wars', 'tagline': 'A long time ago in a galaxy far, far away...', 'runtime': 121, 'release_date': '1977-05-25', 'genres_texte': 'Science Fiction|Action|Adventure', 'tags_texte': 'action|classic|classic sci-fi|darth vader|engrossing adventure|epic|great soundtrack|luke skywalker|nerd|oldie but goodie|robots and androids|sci-fi|space|space action|space adventure|space epic|space opera|star wars', 'casting_texte': 'George Lucas|Mark Hamill|Harrison Ford|Carrie Fisher|Peter Cushing|Alec Guinness', 'actors_texte': 'Mark Hamill|Harrison Ford|Carrie Fisher|Peter Cushing|Alec Guinness', 'directors_texte': 'George Lucas'}
1973 film(s) chargé(s).


,id,title,tagline,runtime,release_date,genres_texte,tags_texte,casting_texte,actors_texte,directors_texte
0,11,Star Wars,"A long time ago in a galaxy far, far away...",121,1977-05-25,Science Fiction|Action|Adventure,action|classic|classic sci-fi|darth vader|engr...,George Lucas|Mark Hamill|Harrison Ford|Carrie ...,Mark Hamill|Harrison Ford|Carrie Fisher|Peter ...,George Lucas
1,12,Finding Nemo,There are 3.7 trillion fish in the ocean. They...,100,2003-05-30,Adventure|Family|Animation,disney|fish|heartwarming|disney|fish|heartwarming,Alexander Gould|Albert Brooks|Ellen DeGeneres|...,Alexander Gould|Albert Brooks|Ellen DeGeneres|...,NaN
2,13,Forrest Gump,The world will never be the same once you've s...,142,1994-06-23,Comedy|Romance|Drama,bittersweet|bubba gump shrimp|emotional|heartw...,Robert Zemeckis|Tom Hanks|Robin Wright|Gary Si...,Tom Hanks|Robin Wright|Gary Sinise|Mykelti Wil...,Robert Zemeckis
3,14,American Beauty,... look closer,122,1999-09-15,Drama,NaN,Sam Mendes|Annette Bening|Kevin Spacey|Thora B...,Annette Bening|Kevin Spacey|Thora Birch|Wes Be...,Sam Mendes
4,18,The Fifth Element,There is no future without it.,126,1997-05-02,Science Fiction|Action|Adventure,NaN,Bruce Willis|Milla Jovovich|Gary Oldman|Ian Ho...,Bruce Willis|Milla Jovovich|Gary Oldman|Ian Ho...,NaN


## 2. Construire le "soup" de texte par film

Deux précautions :
- les entités multi-mots (genres, noms de personnes, certains tags) doivent être collées sans espace (`"Science Fiction"` → `"sciencefiction"`, `"Tom Hanks"` → `"tomhanks"`) — sinon la vectorisation les découpe en tokens séparés et crée de fausses proximités entre films qui partagent juste un mot courant
- un champ vide (`None`) doit devenir une chaîne vide, pas planter la concaténation

In [14]:
def coller_entites(texte):
    """
    Transforme une chaîne d'entités séparées par '|' en tokens sans espace.
    Ex. "Science Fiction|Action" -> "sciencefiction action"

    pd.isna() est utilisé plutôt que "if not texte" : une valeur manquante
    lue depuis une colonne pandas peut être NaN (float), pas None, et NaN
    est "vrai" en contexte booléen (`bool(float('nan'))` vaut True) — donc
    "if not texte" laisserait passer un NaN vers .split() et ferait planter.
    """
    if pd.isna(texte) or not texte:
        return ""
    entites = texte.split("|")
    return " ".join(entite.replace(" ", "").lower() for entite in entites if entite)


def construire_soup(ligne):
    """Concatène genres + tags + casting + tagline en un seul texte par film."""
    genres = coller_entites(ligne["genres_texte"])
    tags = coller_entites(ligne["tags_texte"])
    casting = coller_entites(ligne["casting_texte"])
    tagline = "" if pd.isna(ligne["tagline"]) else str(ligne["tagline"]).lower()

    return " ".join([genres, tags, casting, tagline]).strip()


df["soup"] = df.apply(construire_soup, axis=1)
df[["title", "soup"]].head()

,title,soup
0,Star Wars,sciencefiction action adventure action classic...
1,Finding Nemo,adventure family animation disney fish heartwa...
2,Forrest Gump,comedy romance drama bittersweet bubbagumpshri...
3,American Beauty,drama sammendes annettebening kevinspacey tho...
4,The Fifth Element,sciencefiction action adventure brucewillis m...


### Point de vigilance : films sans contenu exploitable

Un film sans genre, sans tag, sans casting connu et sans tagline aura un "soup" vide — il obtiendra un vecteur quasi nul et ne sera jamais recommandé de façon pertinente. Bon à savoir avant d'interpréter des résultats bizarres sur certains films peu renseignés.

In [6]:
nb_soup_vide = (df["soup"].str.strip() == "").sum()
print(f"{nb_soup_vide} film(s) sans aucune donnée exploitable pour la similarité, sur {len(df)}.")

2 film(s) sans aucune donnée exploitable pour la similarité, sur 1973.


## 3. Vectorisation TF-IDF et similarité cosinus

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer(stop_words="english")
matrice_tfidf = vectorizer.fit_transform(df["soup"])

print(f"Matrice TF-IDF : {matrice_tfidf.shape[0]} films × {matrice_tfidf.shape[1]} termes")

matrice_similarite = cosine_similarity(matrice_tfidf)
print(f"Matrice de similarité : {matrice_similarite.shape}")

Matrice TF-IDF : 1973 films × 8648 termes
Matrice de similarité : (1973, 1973)


### Inspecter `matrice_tfidf`

C'est une matrice **creuse** (`scipy.sparse`) : chaque film n'utilise qu'une petite fraction du vocabulaire total, donc la très grande majorité des cellules valent 0. Une matrice creuse ne stocke que les valeurs non nulles — bien plus économe en mémoire qu'un tableau dense de cette taille.

In [16]:
print(f"Type : {type(matrice_tfidf)}")
print(f"Dimensions : {matrice_tfidf.shape[0]} films × {matrice_tfidf.shape[1]} termes du vocabulaire")

nb_valeurs_non_nulles = matrice_tfidf.nnz
nb_valeurs_totales = matrice_tfidf.shape[0] * matrice_tfidf.shape[1]
taux_remplissage = nb_valeurs_non_nulles / nb_valeurs_totales * 100

print(f"Valeurs non nulles : {nb_valeurs_non_nulles:,} sur {nb_valeurs_totales:,} possibles")
print(f"Taux de remplissage : {taux_remplissage:.2f} % (le reste, ce sont des 0 non stockés)")

Type : <class 'scipy.sparse._csr.csr_matrix'>
Dimensions : 1973 films × 8648 termes du vocabulaire
Valeurs non nulles : 24,710 sur 17,062,504 possibles
Taux de remplissage : 0.14 % (le reste, ce sont des 0 non stockés)


In [18]:
# Le vocabulaire appris : un terme -> une colonne de la matrice
vocabulaire = vectorizer.get_feature_names_out()
print(f"Taille du vocabulaire : {len(vocabulaire)} termes")
print("Exemples de termes :", list(vocabulaire[:50]))

Taille du vocabulaire : 8648 termes
Exemples de termes : ['000', '007', '06oscarnominatedbestmovie', '10', '100', '11', '12', '13', '130', '15', '17', '1700', '18', '1977', '1990s', '1997', '2008', '2027', '21', '23', '24th', '25', '250', '27', '28', '298', '30', '300', '40', '400', '42', '4th', '50', '500', '50cent', '5500', '5th', '60', '65', '700', 'aal', 'aamirkhan', 'aardman', 'aaronblaise', 'aaronburns', 'aaronchen', 'aaroneckhart', 'aaronharris', 'aaronhilmer', 'aaronlohr']


Pour un film donné, on peut regarder quels termes pèsent le plus dans son vecteur — c'est littéralement ce que la similarité cosinus compare ensuite entre deux films.

In [19]:
candidats = df[df["title"].str.contains("forrest gump", case=False, na=False)]
film_exemple = candidats.iloc[0] if not candidats.empty else df.iloc[0]
index_par_id = pd.Series(df.index, index=df["id"])

def termes_importants(film_id, top_n=30):
    """Affiche les termes au poids TF-IDF le plus élevé pour un film donné."""
    idx = index_par_id[film_id]
    ligne_vecteur = matrice_tfidf[idx].toarray().flatten()  # dé-densifie UNE seule ligne

    poids = pd.Series(ligne_vecteur, index=vocabulaire)
    return poids[poids > 0].sort_values(ascending=False).head(top_n)


termes_importants(film_exemple["id"])

bubbagumpshrimp         0.320047
stupidisasstupiddoes    0.320047
shrimp                  0.320047
lieutenantdan           0.320047
touching                0.282901
vietnam                 0.275510
heartwarming            0.269261
bittersweet             0.259073
emotional               0.254801
forrest                 0.160024
gump                    0.160024
eyes                    0.151805
garysinise              0.151805
mykeltiwilliamson       0.151805
sallyfield              0.145974
robinwright             0.131924
seen                    0.127401
robertzemeckis          0.123705
ve                      0.114390
tomhanks                0.105819
world                   0.083550
romance                 0.062647
comedy                  0.045201
drama                   0.044375
dtype: float64

### Inspecter `matrice_similarite`

Ici, contrairement à `matrice_tfidf`, on a une matrice **dense** carrée : `cosine_similarity` compare chaque film à chaque autre, donc `n_films × n_films`, sans creux à exploiter (presque toutes les paires ont une similarité non nulle, même faible).

In [23]:
import numpy as np

print(f"Type : {type(matrice_similarite)}")
print(f"Dimensions : {matrice_similarite.shape}")

# La matrice doit être symétrique : similarité(A, B) == similarité(B, A)
print("Symétrique :", np.allclose(matrice_similarite, matrice_similarite.T))

# La diagonale doit valoir 1.0 : un film est parfaitement similaire à lui-même
print("Diagonale (5 premières valeurs) :", np.diag(matrice_similarite)[:5])

Type : <class 'numpy.ndarray'>
Dimensions : (1973, 1973)
Symétrique : True
Diagonale (5 premières valeurs) : [1. 1. 1. 1. 1.]


In [21]:
# Distribution des scores de similarité (hors diagonale, qui vaut toujours 1.0)
masque_hors_diagonale = ~np.eye(matrice_similarite.shape[0], dtype=bool)
scores_hors_diagonale = matrice_similarite[masque_hors_diagonale]

pd.Series(scores_hors_diagonale).describe()

count    3.890756e+06
mean     1.018063e-02
std      1.885780e-02
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      1.267791e-02
max      8.894937e-01
dtype: float64

Une moyenne proche de 0 est attendue : la plupart des paires de films ne partagent presque aucun terme (genres/casting/tags très différents). Ce qui nous intéresse pour la recommandation, ce sont les quelques scores élevés par film — exactement ce que `recommander()` va chercher juste après.

In [22]:
candidats = df[df["title"].str.contains("forrest gump", case=False, na=False)]
film_exemple = candidats.iloc[0] if not candidats.empty else df.iloc[0]
index_par_id = pd.Series(df.index, index=df["id"])

# Similarité entre deux films précis, lue directement dans la matrice
idx_a = index_par_id[film_exemple["id"]]
un_autre_film = df.iloc[0] if df.iloc[0]["id"] != film_exemple["id"] else df.iloc[1]
idx_b = index_par_id[un_autre_film["id"]]

print(f"{film_exemple['title']} <-> {un_autre_film['title']}")
print(f"Score de similarité : {matrice_similarite[idx_a, idx_b]:.3f}")

Forrest Gump <-> Star Wars
Score de similarité : 0.000


## 4. Fonction de recommandation

In [25]:
index_par_id = pd.Series(df.index, index=df["id"])


def recommander(film_id, top_n=TOP_N):
    """
    Retourne les top_n films les plus proches d'un film donné (par id TMDB),
    triés par score de similarité décroissant. Le film lui-même est exclu.
    """
    if film_id not in index_par_id:
        print(f"Film {film_id} introuvable dans le jeu de données.")
        return pd.DataFrame(columns=["id", "title", "score_similarite"])

    idx = index_par_id[film_id]
    scores = list(enumerate(matrice_similarite[idx]))
    scores = sorted(scores, key=lambda paire: paire[1], reverse=True)
    scores = [(i, s) for i, s in scores if i != idx][:top_n]

    indices_recommandes = [i for i, _ in scores]
    resultat = df.iloc[indices_recommandes][["id", "title"]].copy()
    resultat["score_similarite"] = [round(score, 3) for _, score in scores]

    return resultat.reset_index(drop=True)

## 5. Démo

In [26]:
candidats = df[df["title"].str.contains("forrest gump", case=False, na=False)]
film_exemple = candidats.iloc[0] if not candidats.empty else df.iloc[0]

print(f"Film choisi : {film_exemple['title']} (id {film_exemple['id']})")
recommander(film_exemple["id"])

Film choisi : Forrest Gump (id 13)


,id,title,score_similarite
0,637,Life Is Beautiful,0.162
1,14160,Up,0.159
2,600,Full Metal Jacket,0.129
3,1397385,Little Brother,0.102
4,12,Finding Nemo,0.101


In [27]:
# À vous : changez le titre recherché pour tester sur un autre film de votre base
candidats = df[df["title"].str.contains("father", case=False, na=False)]
if not candidats.empty:
    film_test = candidats.iloc[0]
    print(f"Film choisi : {film_test['title']} (id {film_test['id']})")
    display(recommander(film_test["id"]))
else:
    print("Aucun film trouvé avec ce titre — merci de changer le titre recherché.")

Film choisi : The Godfather (id 238)


,id,title,score_similarite
0,240,The Godfather Part II,0.489
1,242,The Godfather Part III,0.397
2,769,GoodFellas,0.303
3,28,Apocalypse Now,0.198
4,524,Casino,0.194


---
**Pour aller plus loin (pas fait ici) :** on a donné le même poids à genres/tags/casting/tagline en les mettant bout à bout dans un seul texte. Une évolution possible est de construire une matrice par type de feature (genres, casting, tags...) et de les combiner avec des poids différents avant de calculer la similarité — plus précis, mais plus complexe à régler (quel poids pour quoi ?). Bonne piste de discussion/évolution pour la suite du projet plutôt qu'à traiter aujourd'hui.